# Proyecto de Sistemas de Manufactura - Panadería "La Artesanal"
**Consolidado Analítico: Fases 3, 4 y 5 (SPC, Machine Learning, IIoT/MQTT, KPIs Odoo y Sostenibilidad)**

Este Notebook integra en un único flujo ejecutable todos los módulos analíticos desarrollados para el **Horno Principal (Centro de Trabajo 2)** durante la elaboración de la *Torta de Chocolate Artesanal (PT-001)*:
1. **Entrenamiento del Modelo Predictivo (`RandomForestClassifier`)** y generación de artefactos `.pkl`.
2. **Adquisición de Telemetría IIoT (Estructura JSON / Flujo MQTT)** y persistencia en `historial_mqtt_horno.csv`.
3. **Fase 5.1 - Aplicación de SPC:** Gráficos de control X-Individual e índices de capacidad $Cp$ y $Cpk$.
4. **Fase 5.1 - Modelo de Predicción y KPIs Clave:** Proyección de riesgo a +5 lotes, explicabilidad (*Feature Importance*) e indicadores OEE/Costos sincronizados con **Odoo ERP**.
5. **Fase 5.2 - Análisis de Sostenibilidad:** Huella de carbono (Alcance 2 y 3), Economía Circular y Eficiencia Energética.

## 1. Importación de Librerías y Parámetros Base de Odoo ERP

In [ ]:
import os
import json
import random
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Parámetros configurados en Odoo ERP (Fase 2)
COSTO_HORA_HORNO_CS = 90.00       # Costo horario del CT-02 Horno en Odoo (C$/h)
COSTO_ESTANDAR_TORTA_CS = 310.23  # Costo estándar BOM Torta PT-001 en Odoo (C$)
META_OEE_ODOO_PCT = 90.0          # Meta de eficiencia de equipos en Odoo (%)
TIEMPO_CICLO_HORAS = 0.5          # 30 minutos por lote de cocción
POTENCIA_NOMINAL_HORNO_KW = 5.0   # Potencia nominal del horno (5 kW)
POTENCIA_MEZCLADO_KW = 1.5        # Potencia Área de Mezclado CT-01
TARIFA_KWH_CS = 8.50              # Tarifa eléctrica industrial referencial (C$/kWh)
FACTOR_EMISION_RED_CO2 = 0.42     # Factor de emisión red eléctrica (kg CO2eq/kWh)
HUELLA_BOM_INSUMOS_CO2 = 3.85     # Huella Alcance 3 de los 13 insumos del BOM (kg CO2eq/ud)

print('[OK] Librerías y constantes de Odoo ERP cargadas correctamente.')

## 2. Entrenamiento del Modelo de Machine Learning y Exportación `.pkl` (Fase 3)

In [ ]:
np.random.seed(42)
n_samples = 1000

temperatura = np.random.normal(180, 15, n_samples)
horas_uso = np.random.uniform(1, 24, n_samples)
vibracion = np.random.normal(2.5, 0.5, n_samples)

probabilidad_falla = ((temperatura > 200) & (horas_uso > 18)) | (vibracion > 3.5)
falla = np.where(probabilidad_falla, 1, 0)

df_entrenamiento = pd.DataFrame({
    'Temperatura_C': temperatura,
    'Horas_Uso_Continuo': horas_uso,
    'Vibracion_mm_s': vibracion,
    'Falla_Inminente': falla
})

X = df_entrenamiento[['Temperatura_C', 'Horas_Uso_Continuo', 'Vibracion_mm_s']]
y = df_entrenamiento['Falla_Inminente']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_rf.fit(X_train_scaled, y_train)
y_pred = modelo_rf.predict(X_test_scaled)

print('Matriz de Confusión:\n', confusion_matrix(y_test, y_pred))
print('\nReporte de Clasificación:\n', classification_report(y_test, y_pred))

joblib.dump(modelo_rf, 'modelo_mantenimiento_horno.pkl')
joblib.dump(scaler, 'escalador_horno.pkl')
print('[OK] Artefactos modelo_mantenimiento_horno.pkl y escalador_horno.pkl guardados.')

## 3. Adquisición de Telemetría IIoT / MQTT y Carga de Historial (Fase 4)

In [ ]:
ARCHIVO_CSV_MQTT = 'historial_mqtt_horno.csv'

# Si existe el historial real de Mosquitto se carga; si se ejecuta en la nube desde cero, genera 80 lotes
if os.path.exists(ARCHIVO_CSV_MQTT):
    df_historial = pd.read_csv(ARCHIVO_CSV_MQTT)
    print(f'[OK] Repositorio MQTT cargado desde {ARCHIVO_CSV_MQTT} ({len(df_historial)} lotes registrados).')
else:
    print('[INFO] Generando simulación local de tramas MQTT para entorno autónomo...')
    random.seed(42)
    registros = []
    horas_op = 10.0
    for lote in range(1, 81):
        es_anomalia = (lote % 4 == 0)
        t_c = round(random.uniform(202.0, 218.0), 2) if es_anomalia else round(random.gauss(180.0, 4.5), 2)
        h_u = round(random.uniform(18.5, 23.0), 2) if es_anomalia else round(horas_op, 2)
        v_m = round(random.uniform(3.6, 4.7), 2) if es_anomalia else round(random.gauss(2.5, 0.3), 2)
        df_in = pd.DataFrame({'Temperatura_C': [t_c], 'Horas_Uso_Continuo': [h_u], 'Vibracion_mm_s': [v_m]})
        in_esc = scaler.transform(df_in)
        pred = int(modelo_rf.predict(in_esc)[0])
        prob = round(float(modelo_rf.predict_proba(in_esc)[0][1]) * 100.0, 1)
        registros.append({
            'id_lote': lote, 'timestamp': f'2026-10-16 21:00:{lote % 60:02d}',
            'Temperatura_C': t_c, 'Horas_Uso_Continuo': h_u, 'Vibracion_mm_s': v_m,
            'Falla_Inminente': pred, 'Probabilidad_Pct': prob,
            'Estado_Equipo': 'ALERTA CRÍTICA' if pred == 1 else 'ESTADO OPTIMO'
        })
        horas_op += 0.25
    df_historial = pd.DataFrame(registros)
    df_historial.to_csv(ARCHIVO_CSV_MQTT, index=False)

df_ventana = df_historial.tail(60).copy()
df_ventana.tail(5)

## 4. Fase 5.1: Aplicación de SPC (Gráficos de Control, $Cp$ y $Cpk$)

In [ ]:
def evaluar_spc_horno(df_datos, lei=165.0, les=195.0, filtrar_anomalias=False, titulo_escenario=''):
    df_spc = df_datos.copy()
    if filtrar_anomalias:
        df_spc = df_spc[df_spc['Temperatura_C'] <= 200.0]

    serie = df_spc['Temperatura_C']
    lotes = df_spc['id_lote']
    media = serie.mean()
    sigma = serie.std()
    lcs, lci = media + 3 * sigma, media - 3 * sigma
    cp = (les - lei) / (6 * sigma)
    cpk = min((les - media) / (3 * sigma), (media - lei) / (3 * sigma))
    fuera_esp = ((serie > les) | (serie < lei)).sum()

    estado = 'PROCESO CAPAZ (Cpk >= 1.33)' if cpk >= 1.33 else ('MARGINAL (1.00 <= Cpk < 1.33)' if cpk >= 1.0 else 'NO CAPAZ (Cpk < 1.00)')
    print(f'=== {titulo_escenario} ===')
    print(f'Media: {media:.2f} °C | Sigma: {sigma:.2f} °C | LCI/LCS: {lci:.1f}/{lcs:.1f} °C')
    print(f'Índice Cp: {cp:.2f} | Índice Cpk: {cpk:.2f} | Fuera de Especificación: {fuera_esp} lotes -> [{estado}]\n')

    fig, ax = plt.subplots(figsize=(11, 3.8))
    ax.plot(lotes, serie, marker='o', color='#1f77b4', markersize=4, label='Temperatura (°C)')
    ax.axhline(media, color='green', linewidth=1.5, label=f'Media ({media:.1f}°C)')
    ax.axhline(lcs, color='red', linestyle='--', label=f'LCS (+3σ = {lcs:.1f}°C)')
    ax.axhline(lci, color='red', linestyle='--', label=f'LCI (-3σ = {lci:.1f}°C)')
    ax.axhline(les, color='darkorange', linestyle=':', linewidth=1.5, label=f'LES ({les:.1f}°C)')
    ax.axhline(lei, color='darkorange', linestyle=':', linewidth=1.5, label=f'LEI ({lei:.1f}°C)')
    anom = df_spc[(df_spc['Temperatura_C'] > les) | (df_spc['Temperatura_C'] < lei)]
    if not anom.empty:
        ax.scatter(anom['id_lote'], anom['Temperatura_C'], color='red', s=55, zorder=5, label='Fuera de Esp.')
    ax.set_title(f'Gráfico de Control SPC - {titulo_escenario}')
    ax.set_xlabel('ID de Lote')
    ax.set_ylabel('Temperatura (°C)')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper right', fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()

# Escenario 1: Con anomalías inducidas
evaluar_spc_horno(df_ventana, lei=165.0, les=195.0, filtrar_anomalias=False, titulo_escenario='Escenario 1: Con Perturbaciones Inducidas')
# Escenario 2: Régimen nominal aislando picos de prueba
evaluar_spc_horno(df_ventana, lei=160.0, les=200.0, filtrar_anomalias=True, titulo_escenario='Escenario 2: Régimen Nominal Controlado')

## 5. Fase 5.1: Modelo de Predicción y Dashboard de Indicadores Clave (KPIs Planta - Odoo)

In [ ]:
total_lotes = len(df_ventana)
lotes_falla = int((df_ventana['Falla_Inminente'] == 1).sum())
lotes_optimos = total_lotes - lotes_falla
disponibilidad_pct = (lotes_optimos / total_lotes) * 100.0
desv_vib = np.abs(df_ventana['Vibracion_mm_s'] - 2.5).mean()
rendimiento_pct = float(np.clip(100.0 - (desv_vib * 5.0), 75.0, 99.5))
lotes_calidad_ok = int(((df_ventana['Temperatura_C'] >= 165.0) & (df_ventana['Temperatura_C'] <= 195.0)).sum())
calidad_pct = (lotes_calidad_ok / total_lotes) * 100.0
oee_global_pct = (disponibilidad_pct / 100.0) * (rendimiento_pct / 100.0) * (calidad_pct / 100.0) * 100.0
costo_op_cs = total_lotes * TIEMPO_CICLO_HORAS * COSTO_HORA_HORNO_CS
riesgo_merma_cs = (total_lotes - lotes_calidad_ok) * COSTO_ESTANDAR_TORTA_CS

print(f'OEE Global Horno: {oee_global_pct:.1f}% (Meta Odoo: {META_OEE_ODOO_PCT}%) | Disponibilidad: {disponibilidad_pct:.1f}% | Calidad FPY: {calidad_pct:.1f}%')
print(f'Costo Operativo Horno: C$ {costo_op_cs:,.2f} | Riesgo Merma (PT-001): C$ {riesgo_merma_cs:,.2f}\n')

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4.0))

# Gráfico 1: Proyección de Probabilidad de Falla (+5 lotes)
lx, py = df_ventana['id_lote'].values, df_ventana['Probabilidad_Pct'].values
ax1.plot(lx, py, marker='o', color='#d62728', markersize=3.5, label='Prob. Falla ML (%)')
ax1.axhline(50.0, color='black', linestyle='--', label='Umbral Crítico (50%)')
pol = np.poly1d(np.polyfit(lx, py, 1))
lfut = np.arange(lx[-1] + 1, lx[-1] + 6)
ax1.plot(np.concatenate([[lx[-1]], lfut]), np.concatenate([[py[-1]], np.clip(pol(lfut), 0, 100)]), 'x:', color='purple', label='Proyección (+5 lotes)')
ax1.set_title('Inferencia y Proyección Random Forest')
ax1.set_xlabel('ID de Lote')
ax1.set_ylabel('Probabilidad (%)')
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(fontsize=7.5)

# Gráfico 2: Feature Importance
vars_ml = ['Temperatura (°C)', 'Horas Uso', 'Vibración (mm/s)']
imp = modelo_rf.feature_importances_ * 100.0
ax2.barh(vars_ml, imp, color=['#ff7f0e', '#1f77b4', '#2ca02c'], height=0.5)
ax2.set_xlim(0, 100)
ax2.set_title('Importancia de Variables (Feature Importance)')
ax2.set_xlabel('Peso Relativo (%)')
ax2.grid(True, axis='x', linestyle='--', alpha=0.5)

# Gráfico 3: Pilares OEE vs Meta Odoo 90%
cats = ['Disponib.', 'Rendim.', 'Calidad', 'OEE Global']
vals = [disponibilidad_pct, rendimiento_pct, calidad_pct, oee_global_pct]
ax3.bar(cats, vals, color=['#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd'], width=0.5)
ax3.axhline(META_OEE_ODOO_PCT, color='red', linestyle='--', label='Meta Odoo (90%)')
ax3.set_ylim(0, 115)
ax3.set_title('Desglose OEE vs. Meta Odoo ERP')
ax3.legend(fontsize=8)
ax3.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 6. Fase 5.2: Análisis de Sostenibilidad (Huella de Carbono, Circularidad y Eficiencia Energética)

In [ ]:
factor_termico = df_ventana['Temperatura_C'] / 180.0
penal_vib = np.where(df_ventana['Vibracion_mm_s'] > 3.0, 1.12, 1.0)
df_ventana['KWh_Actual'] = (POTENCIA_NOMINAL_HORNO_KW * factor_termico * penal_vib * TIEMPO_CICLO_HORAS) + (POTENCIA_MEZCLADO_KW * (37.0/60.0)) + 0.15
df_ventana['KWh_Optimizado'] = (POTENCIA_NOMINAL_HORNO_KW * TIEMPO_CICLO_HORAS * 0.84) + ((POTENCIA_MEZCLADO_KW * (37.0/60.0) + 0.15) * 0.92)

kwh_act = df_ventana['KWh_Actual'].sum()
kwh_opt = df_ventana['KWh_Optimizado'].sum()
ahorro_kwh = kwh_act - kwh_opt
co2_alcance2 = kwh_act * FACTOR_EMISION_RED_CO2
huella_unitaria = ((kwh_opt * FACTOR_EMISION_RED_CO2) + (total_lotes * HUELLA_BOM_INSUMOS_CO2 * 0.91)) / total_lotes

print(f'Consumo Eléctrico: {kwh_act:.1f} kWh | Ahorro con Eficiencia IIoT/MRP: {ahorro_kwh:.1f} kWh (C$ {ahorro_kwh * TARIFA_KWH_CS:,.2f})')
print(f'Huella Alcance 2: {co2_alcance2:.2f} kg CO2e | Huella Unitaria Optimizada (PT-001): {huella_unitaria:.2f} kg CO2e/torta\n')

fig, (ax_c, ax_e) = plt.subplots(1, 2, figsize=(13, 3.8))
fuentes = ['Lácteos BOM', 'Cacao y Choc.', 'Horno CT-02 (Elec.)', 'Harina/Azúcar/Huevos', 'Mezclado/Dec.']
emisiones = [1.65, 1.35, co2_alcance2 / total_lotes, 0.85, 0.45]
ax_c.barh(fuentes, emisiones, color=['#2ca02c', '#8c564b', '#d62728', '#ffbb00', '#1f77b4'])
ax_c.set_title('Desglose de Huella de Carbono (kg CO2eq / Torta PT-001)')
ax_c.set_xlabel('kg CO2eq')
ax_c.grid(True, axis='x', linestyle='--', alpha=0.5)

ax_e.plot(df_ventana['id_lote'], df_ventana['KWh_Actual'], 'o-', color='#d62728', markersize=3, label='Consumo Real (kWh/lote)')
ax_e.plot(df_ventana['id_lote'], df_ventana['KWh_Optimizado'], '--', color='#2ca02c', linewidth=1.8, label='Consumo Optimizado IIoT (kWh/lote)')
ax_e.fill_between(df_ventana['id_lote'], df_ventana['KWh_Optimizado'], df_ventana['KWh_Actual'], color='#2ca02c', alpha=0.2, label='Energía Ahorrada')
ax_e.set_title('Eficiencia Energética: Consumo Real vs. Optimizado')
ax_e.set_xlabel('ID de Lote')
ax_e.set_ylabel('kWh')
ax_e.legend(fontsize=8)
ax_e.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()